# Streaming, Performance, Spark Connect & Pandas API Cheatsheet

**Exam Domains: Streaming (10%), Troubleshooting (10%),**

              Spark Connect (5%), Pandas API (5%)

## STRUCTURED STREAMING

In [ ]:
# Read a stream — schema MUST be explicit (no inferSchema for streaming)
stream_df = (
    spark.readStream
    .schema(my_schema)           # required
    .json("path")                # or .parquet(), .format("delta"), etc.
)

# Write a stream — checkpoint is REQUIRED
query = (
    stream_df
    .writeStream
    .format("delta")
    .outputMode("append")        # "append", "update", or "complete"
    .option("checkpointLocation", "/path/to/checkpoint")
    .trigger(availableNow=True)  # or processingTime="10 seconds"
    .toTable("output_table")
)

# Output modes:
#   append   — new rows only (default, no aggregation or with watermark)
#   update   — changed rows (aggregations, updates existing keys)
#   complete — full result table (aggregations, replaces all output)

# Trigger types:
#   availableNow=True          — process all data, then stop (modern)
#   processingTime="10 seconds" — micro-batch every 10s (continuous)
#   once=True                  — deprecated, use availableNow

# Monitor active streams
spark.streams.active            # list of active queries
query.status                    # current status
query.awaitTermination()        # block until stream stops
query.stop()                    # stop the stream

# NOT supported on streaming DataFrames: count(), show(), collect()

## CACHING & PERSISTENCE

In [ ]:
from pyspark import StorageLevel

df.cache()                      # lazy — PySpark default: MEMORY_AND_DISK
df.persist(StorageLevel.MEMORY_AND_DISK)   # explicit level
df.persist(StorageLevel.DISK_ONLY)         # disk only
df.persist(StorageLevel.MEMORY_ONLY)       # memory only, drop if full
df.unpersist()                  # remove from cache
df.is_cached                    # bool — check if cached

# cache() is LAZY — caching happens on the next action, not immediately

## PARTITIONING

In [ ]:
# repartition — full shuffle, increase OR decrease partitions
df.repartition(8)               # by number
df.repartition(8, "col")        # by column (same key -> same partition)

# coalesce — NO full shuffle, can ONLY decrease partitions
df.coalesce(2)

df.rdd.getNumPartitions()       # check partition count

## BROADCAST JOINS

In [ ]:
from pyspark.sql.functions import broadcast

# Broadcast small table to all executors (avoids shuffle)
large_df.join(broadcast(small_df), on="key")

# Auto-broadcast threshold (default 10MB)
# spark.conf.get("spark.sql.autoBroadcastJoinThreshold")

## EXECUTION PLANS

In [ ]:
df.explain()                    # physical plan only
df.explain(True)                # parsed -> analyzed -> optimized -> physical
df.explain("formatted")         # human-readable formatted plan

# Read bottom-to-top. Key operators:
#   Scan        — reading from source
#   Filter      — predicate pushdown
#   Exchange    — shuffle (data redistribution)
#   HashAggregate — aggregation
#   BroadcastHashJoin — broadcast join (no shuffle)
#   SortMergeJoin — shuffle join

## ADAPTIVE QUERY EXECUTION (AQE)

In [ ]:
# AQE re-optimizes at runtime. Key features:
#   - Coalesces small shuffle partitions
#   - Switches to broadcast join if runtime size < threshold
#   - Handles skewed joins

# spark.conf.get("spark.sql.adaptive.enabled")  # default: True
# spark.conf.get("spark.sql.adaptive.coalescePartitions.enabled")

## SPARK CONNECT

In [ ]:
# Thin client protocol — decouples client from cluster via gRPC
# Same DataFrame API — no code changes needed

# Remote connection (from outside Databricks):
# from pyspark.sql import SparkSession
# spark = SparkSession.builder.remote("sc://hostname:443/;token=TOKEN").getOrCreate()

# In Databricks serverless, Spark Connect is used by default

## PANDAS API ON SPARK

In [ ]:
import pyspark.pandas as ps

# Read — runs distributed, NOT on driver
psdf = ps.read_table("my_table")
psdf = ps.DataFrame({"a": [1, 2], "b": [3, 4]})

# pandas-style operations — distributed under the hood
psdf.head()
psdf.describe()
psdf.groupby("col").mean()
psdf["col"].value_counts()

# Convert between types
spark_df = psdf.to_spark()        # pandas-on-Spark -> Spark DataFrame
psdf = spark_df.pandas_api()      # Spark DataFrame -> pandas-on-Spark

# KEY DIFFERENCE:
#   pyspark.pandas  -> distributed on cluster (safe for large data)
#   df.toPandas()   -> collects ALL data to driver (OOM risk!)